In [ ]:
%cd .. 
import os
import pandas as pd
from dotenv import load_dotenv
from scraper import run
from scraper.sources import ACTOR_NAMES
import argparse

load_dotenv()
assert os.environ.get("APIFY_TOKEN"), "Set APIFY_TOKEN in .env"
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env"

# Berlin-based, German market
config_dict = {
    "country": "de",
    "location": "Berlin",
    "limit": 50,
    "date_posted": "7",
    "platforms": ACTOR_NAMES, # run across all platforms simultaneously
    "discipline": "",
    "max_pages": 20,
}

## Define the search portfolio
Each tuple is (label, query). The label becomes a column so you can see which search a row came from.

In [ ]:
SEARCHES = [
    # ── Track 1: Core ML/AI engineering ──────────────────────────────
    ("applied_ml",         "applied machine learning engineer"),
    ("research_engineer",  "AI research engineer"),
    ("llm_engineer",       "LLM engineer agentic"),
    # ("rag_engineer",       "retrieval augmented generation RAG"),
    # ("genai_engineer",     "generative AI engineer"),

    # ── Track 2: Data science / analytics ────────────────────────────
    ("data_scientist",     "data scientist"),
    ("data_analyst",       "data analyst"),
    # ("product_analyst",    "product analyst"),
    # ("business_analyst",   "business intelligence analyst"),
    ("quant_analyst",      "quantitative analyst"),
    ("applied_research",   "applied researcher"),

    # ── Track 3: Audio / speech / MIR niche ──────────────────────────
    # ("audio_ml",           "audio machine learning"),
    ("speech_ml",          "speech machine learning"),
    # ("music_ml",           "music machine learning"),
    ("music_tech",         "music technology engineer"),

    # ── Track 4: Research / academic ─────────────────────────────────
    ("postdoc_ai",         "postdoc artificial intelligence"),
    ("wiss_mit_ki",        "Wissenschaftlicher Mitarbeiter KI"),
    ("research_institute", "research scientist machine learning"),

    # ── Track 5: Mission-driven / impact ─────────────────────────────
    ("climate_ai",         "climate data"),
    ("ngo_data",           "data nonprofit charity"),
    ("civic_tech",         "civic tech data"),
    ("data4good",          "data for good"),
]

##  Run all searches
Sequential, with a per-query try/except so one bad search doesn't kill the rest. Stores results in a dict keyed by label so you can re-run individual cells without losing progress.

In [ ]:
results = {}  # label -> DataFrame

for label, query in SEARCHES:
    if label in results:
        print(f"[{label}] cached, skipping")
        continue
    try:
        args = argparse.Namespace(**config_dict)
        args.title = query
        df = run(args)
        df["search_label"] = label
        df["search_query"] = query
        results[label] = df
        print(f"[{label}] {len(df)} jobs")
    except Exception as e:
        print(f"[{label}] FAILED: {e}")
        results[label] = pd.DataFrame()

## Combine and dedupe
The same job will appear under multiple search labels, keep the first occurrence.

In [ ]:
all_jobs = pd.concat(results.values(), ignore_index=True)
print(f"raw rows across all searches: {len(all_jobs)}")

# # drop duplicates 
dedupe_key = ["employer_name", "title", "city"]

jobs = (all_jobs.drop_duplicates(subset=dedupe_key, keep="first")
        .drop(columns=["search_label", "search_query"])
        .reset_index(drop=True))

print(f"unique jobs: {len(jobs)}")
jobs[["title", "employer_name", "city"]].head(20)

## Save
Parquet keeps types and the raw column intact. CSV is for spreadsheet skimming.

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M")
raw_dir = Path(f"raw_results/{stamp}")
raw_dir.mkdir(parents=True, exist_ok=True)

def safe_name(s, n: int = 60) -> str:
    """Filesystem-safe slug. Handles NaN, None, and non-string types."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return "untitled"
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")[:n] or "untitled"

for i, row in jobs.iterrows():
    fname = f"{i:04d}_{safe_name(row['employer_name'])}_{safe_name(row['title'])}.json"
    with open(raw_dir / fname, "w", encoding="utf-8") as f:
        json.dump(row["raw"], f, ensure_ascii=False, indent=2, default=str)

jobs.drop(columns=["raw"]).to_parquet(f"jobs_{stamp}.parquet")
jobs.drop(columns=["raw", "description"]).to_csv(f"jobs_{stamp}.csv", index=False)

print(f"Saved {len(jobs)} raw JSONs to {raw_dir}/")
print(f"Saved jobs_{stamp}.parquet (full minus raw) and jobs_{stamp}.csv (skim)")

In [ ]:
jobs

## Concat and dedupe further dataframes

In [ ]:
# jobs = pd.read_parquet("scraper/jobs_20260508_1147.parquet")
# more_jobs = pd.read_parquet("scraper/jobs_20260504_1545.parquet")

dedupe_key = ["employer_name", "title", "city"]
jobs = (pd.concat([jobs,more_jobs],ignore_index=True)
        .drop_duplicates(subset=dedupe_key, keep="first")
        .reset_index(drop=True))

In [ ]:
len(jobs)

In [ ]:
jobs = jobs.drop(columns=["matched_by", "score", "matched_by_n"])
jobs

In [ ]:
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
jobs.to_parquet(f"jobs_{config_dict["location"]}_{stamp}.parquet")